# FlyOpt Doğrulama Paketi (Colab / GPU)

**2026-09-21.** Bu notebook, FlyOpt projesinin şu anki en kritik açık sorularını GPU üzerinde hızlıca (saatler yerine dakikalar içinde) test eder:

1. **Sağlık kontrolü** — resmi bulguyu (δ=0.884, seed=9000) bu ortamda tekrar üretip pipeline'ın doğru çalıştığını doğrular.
2. **Alt-graf-tohum genelliği** — resmi bulgu tek bir şanslı alt-grafa mı özgü, yoksa dişi FlyWire'ın genel bir özelliği mi?
3. **T-parametresi duyarlılığı** — hem dişi hem erkek connectome'u SİMETRİK olarak farklı T (düşünme süresi) değerleriyle test eder (tek tarafı ayarlayıp p-hacking riskine girmeden).
4. **"Pareto/Foraging" korelasyon iddiasının NULL-KONTROLLÜ doğru testi** — dış bir analizde bulunan (-0.65) korelasyonun gerçekten connectome'a mı özgü, yoksa herhangi bir 2 boyutlu doğrusal okuma katmanının genel bir matematik özelliği mi olduğunu, degree_preserving_rewire null'una karşı test ederek netleştirir. **Bu adım olmadan o korelasyon hiçbir şey kanıtlamaz** — bu notebook'un en önemli düzeltmesi budur.

Sonunda tüm tablolar/grafikler `flyopt_verification_results/` klasörüne kaydedilip `flyopt_verification_results.zip` olarak indirilir.

## Kurulum notları
- Bu repo **özel (private)** bir GitHub deposu — Colab'da klonlamak için bir Personal Access Token gerekir, YA DA repoyu ve `data/processed/` klasörünü zip'leyip manuel yükleyebilirsin (aşağıdaki hücrede iki seçenek de var).
- `data/processed/` altında gerekli dosyalar: `adjacency.npz`, `afferent_indices.npy`, `efferent_indices.npy` (dişi FlyWire) ve `malecns_adjacency.npz`, `malecns_afferent_indices.npy`, `malecns_efferent_indices.npy` (erkek CNS) — bunlar `.gitignore`'da olduğu için git'te YOK, ayrıca yüklemen gerekiyor.

In [ ]:
# --- KURULUM: SEÇENEK A (önerilen, en basit) ---
# Bilgisayarında şunu çalıştır, sonra çıkan zip'i bu hücrenin altında açılacak yükleme kutusuna sürükle:
#   cd C:/projeler/fly_op
#   (PowerShell) Compress-Archive -Path src,data/processed -DestinationPath flyopt_colab_bundle.zip

from google.colab import files
import zipfile, os

print("flyopt_colab_bundle.zip dosyasini sec (src/ ve data/processed/ icermeli):")
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn) as z:
            z.extractall(".")
        print(f"{fn} acildi.")

assert os.path.exists("src/flyopt"), "src/flyopt bulunamadi -- zip icerigini kontrol et"
assert os.path.exists("data/processed/adjacency.npz"), "data/processed/adjacency.npz bulunamadi"

In [ ]:
# --- KURULUM: SEÇENEK B (alternatif, git clone ile) ---
# Yukaridaki hucreyi calistirdiysan bu hucreyi ATLA. GitHub Personal Access Token gerekir.
# GITHUB_TOKEN = ""  # buraya token yapistir
# !git clone https://{GITHUB_TOKEN}@github.com/AutoPyloter/fly_op.git repo
# %cd repo
# # data/processed/ dosyalarini ayrica Drive'dan/yerelden yuklemen gerekecek (git'te yok)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))

!pip install -q networkx pandas scipy matplotlib

import json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import sparse, stats

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

OUT_DIR = "flyopt_verification_results"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
from flyopt.benchmarks_geo import DIM, factor_of_safety, random_geometry
from flyopt.substrates.graph_builders import degree_preserving_rewire, er_null
from flyopt.variants.fly_proposer_scene import _rays, _teacher_delta
from flyopt.variants.rate_brain import (
    RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode, train,
)

DATA_PROCESSED = "data/processed"

def build_training_set_geo(n_problems, n_starts, seed, ray_radius=1.0):
    rng = np.random.default_rng(seed)
    X, Y = [], []
    for _ in range(n_problems):
        geo = random_geometry(rng)
        lo, hi = geo.param_bounds()
        f = lambda p, geo=geo: factor_of_safety(p, geo)
        for _ in range(n_starts):
            x = rng.uniform(lo, hi)
            fx = f(x)
            rays = _rays(x, f, fx, ray_radius)
            target = _teacher_delta(x, rays, ray_radius)
            norm = np.linalg.norm(target)
            target = target / norm if norm > 1e-8 else np.zeros(DIM)
            X.append(rays); Y.append(target)
    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)

def cliffs_delta(real, null):
    gt = np.sum(real[:, None] < null[None, :])
    lt = np.sum(real[:, None] > null[None, :])
    return float((gt - lt) / (len(real) * len(null)))

def run_comparison(base_weights, afferent, efferent, subgraph_seed, n_seeds, T=8, ray_radius=1.0,
                    n_rays=None, n_readout=30, subgraph_size=3000, epochs=300, lr=3e-3):
    """Real vs degree_preserving_rewire, n_seeds pilot, returns dict of stats."""
    n_rays = n_rays or 2 * DIM
    encode_full, decode_full = select_connected_encode_decode(
        base_weights, afferent, efferent, n_encode=n_rays, n_decode=n_readout,
        max_hops=6, n_encode_candidates=200, seed=subgraph_seed,
    )
    sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, subgraph_size, seed=subgraph_seed)
    cfg = RateBrainConfig(dim=DIM, n_readout=len(decode_idx), T=T, ray_radius=ray_radius, decode_scale=0.5, train_gain=True)

    real_list, null_list = [], []
    for seed in range(n_seeds):
        X, Y = build_training_set_geo(20, 10, seed=9500 + seed, ray_radius=ray_radius)
        X_test, Y_test = build_training_set_geo(10, 10, seed=19500 + seed, ray_radius=ray_radius)
        sub_null = degree_preserving_rewire(sub_real, seed=seed)
        for lst, sub in [(real_list, sub_real), (null_list, sub_null)]:
            brain = RateBrain(sub, encode_idx, decode_idx, cfg, seed=seed).to(DEVICE)
            train(brain, X, Y, epochs=epochs, lr=lr)
            with torch.no_grad():
                pred = brain(torch.as_tensor(X_test, device=DEVICE))
                mse = float(((pred - torch.as_tensor(Y_test, device=DEVICE)) ** 2).mean().item())
            lst.append(mse)

    real, null = np.array(real_list), np.array(null_list)
    delta = cliffs_delta(real, null)
    try:
        mw_p = stats.mannwhitneyu(real, null, alternative="two-sided").pvalue
    except ValueError:
        mw_p = float("nan")
    return {
        "subgraph_seed": subgraph_seed, "n": n_seeds, "T": T,
        "real_median": float(np.median(real)), "null_median": float(np.median(null)),
        "cliffs_delta": delta, "mannwhitney_p": float(mw_p),
        "gate_pass": bool(mw_p < 0.05 and abs(delta) > 0.33),
        "real_values": real.tolist(), "null_values": null.tolist(),
    }

print("utility fonksiyonlari hazir")

## 1. Sağlık kontrolü — resmi bulguyu tekrar üret
seed=9000, T=8, degree_preserving_rewire null. Referans: δ=0.884 (n=30, CPU, önceki oturum).

In [ ]:
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/adjacency.npz")
afferent = np.load(f"{DATA_PROCESSED}/afferent_indices.npy")
efferent = np.load(f"{DATA_PROCESSED}/efferent_indices.npy")

t0 = time.time()
sanity = run_comparison(base_weights, afferent, efferent, subgraph_seed=9000, n_seeds=8, T=8)
print(f"sure: {time.time()-t0:.1f}s")
print(json.dumps({k: v for k, v in sanity.items() if k not in ('real_values','null_values')}, indent=2))
print("\nREFERANS (CPU, n=30): delta=0.884, Mann-Whitney p~=0.000000")

## 2. Alt-graf-tohum genelliği taraması
Birden fazla bağımsız alt-graf seçimi (encode/decode seed'i) — δ'nın dağılımını ölçer. Amaç: seed=9000 tipik mi istisna mı?

In [ ]:
SUBGRAPH_SEEDS = list(range(9000, 9021))  # 21 bagimsiz alt-graf secimi -- GPU'da ucuz
sweep_results = []
t0 = time.time()
for s in SUBGRAPH_SEEDS:
    r = run_comparison(base_weights, afferent, efferent, subgraph_seed=s, n_seeds=6, T=8)
    sweep_results.append(r)
    print(f"seed={s}: delta={r['cliffs_delta']:.3f}  p={r['mannwhitney_p']:.3f}  gate={'PASS' if r['gate_pass'] else 'FAIL'}")
print(f"toplam sure: {time.time()-t0:.1f}s")

df_sweep = pd.DataFrame([{k: v for k, v in r.items() if k not in ('real_values','null_values')} for r in sweep_results])
df_sweep.to_csv(f"{OUT_DIR}/subgraph_seed_sweep.csv", index=False)
df_sweep

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2a9d8f" if g else "#e76f51" for g in df_sweep["gate_pass"]]
ax.bar(df_sweep["subgraph_seed"].astype(str), df_sweep["cliffs_delta"], color=colors)
ax.axhline(0.33, color="gray", linestyle="--", linewidth=1, label="gate esigi (|delta|>0.33)")
ax.axhline(-0.33, color="gray", linestyle="--", linewidth=1)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("alt-graf secim tohumu"); ax.set_ylabel("Cliff's delta")
ax.set_title("Alt-graf-tohum genelligi: her tohum bagimsiz bir 3000-noron secimi")
ax.legend(); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/subgraph_seed_sweep.png", dpi=140); plt.show()

print(f"\nOzet: {df_sweep['gate_pass'].sum()}/{len(df_sweep)} tohum gate PASS aldi")
print(f"delta dagilimi: medyan={df_sweep['cliffs_delta'].median():.3f}  min={df_sweep['cliffs_delta'].min():.3f}  max={df_sweep['cliffs_delta'].max():.3f}")

## 3. T-parametresi duyarlılığı — SİMETRİK (dişi VE erkek birlikte)
Tek tarafı ayarlayıp p-hacking riskine girmeden: her T değerinde HEM dişi FlyWire HEM erkek CNS test edilir.

In [ ]:
male_adj_path = f"{DATA_PROCESSED}/malecns_adjacency.npz"
has_male = os.path.exists(male_adj_path)
print("erkek CNS verisi bulundu mu:", has_male)

if has_male:
    male_weights = sparse.load_npz(male_adj_path)
    male_afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
    male_efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

T_VALUES = [4, 8, 12, 16, 20, 24]
t_results = []
t0 = time.time()
for T in T_VALUES:
    r_f = run_comparison(base_weights, afferent, efferent, subgraph_seed=9000, n_seeds=6, T=T)
    r_f["connectome"] = "female_flywire"
    t_results.append(r_f)
    print(f"T={T} female: delta={r_f['cliffs_delta']:.3f} p={r_f['mannwhitney_p']:.3f}")
    if has_male:
        r_m = run_comparison(male_weights, male_afferent, male_efferent, subgraph_seed=9000, n_seeds=6, T=T)
        r_m["connectome"] = "male_cns"
        t_results.append(r_m)
        print(f"T={T} male:   delta={r_m['cliffs_delta']:.3f} p={r_m['mannwhitney_p']:.3f}")
print(f"toplam sure: {time.time()-t0:.1f}s")

df_t = pd.DataFrame([{k: v for k, v in r.items() if k not in ('real_values','null_values')} for r in t_results])
df_t.to_csv(f"{OUT_DIR}/T_sensitivity.csv", index=False)
df_t

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, grp in df_t.groupby("connectome"):
    grp = grp.sort_values("T")
    ax.plot(grp["T"], grp["cliffs_delta"], marker="o", label=name)
ax.axhline(0.33, color="gray", linestyle="--", linewidth=1)
ax.axhline(-0.33, color="gray", linestyle="--", linewidth=1)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("T (ic alt-adim sayisi / dusunme suresi)"); ax.set_ylabel("Cliff's delta")
ax.set_title("T-parametresi duyarliligi: disi vs erkek, simetrik test")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/T_sensitivity.png", dpi=140); plt.show()

## 4. "Pareto/Foraging" korelasyon iddiasının NULL-KONTROLLÜ testi

Dış bir analizde 2 boyutlu ('Enerji'/'Ödül') bir okuma katmanının çıktıları arasında -0.65 korelasyon bulunup "biyolojik bilgelik" olarak yorumlanmıştı. **Ama hiçbir null-model karşılaştırması yapılmamıştı.** Burada AYNI korelasyonu hem gerçek connectome'da hem degree_preserving_rewire null'unda, hem EĞİTİLMEMİŞ (rastgele başlangıç) hem gerekirse eğitilmiş halde ölçüyoruz. Eğer null model de benzer bir korelasyon gösteriyorsa, bu "biyolojik" değil, 2 boyutlu doğrusal okuma katmanının genel bir matematik özelliğidir.

In [ ]:
def pareto_correlation_check(base_weights, afferent, efferent, subgraph_seed=9000, T=20, n_scenarios=10000, seed=0):
    n_rays = 4  # 2*dim rays icin dim=2
    encode_full, decode_full = select_connected_encode_decode(
        base_weights, afferent, efferent, n_encode=n_rays, n_decode=30,
        max_hops=6, n_encode_candidates=200, seed=subgraph_seed,
    )
    sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=subgraph_seed)
    sub_null = degree_preserving_rewire(sub_real, seed=seed)
    cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=T, ray_radius=1.0, decode_scale=0.5, train_gain=True)

    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, size=(n_scenarios, n_rays)).astype(np.float32)

    out = {}
    for name, sub in [("real_connectome", sub_real), ("degree_preserving_null", sub_null)]:
        brain = RateBrain(sub, encode_idx, decode_idx, cfg, seed=seed).to(DEVICE)
        with torch.no_grad():
            pred = brain(torch.as_tensor(X, device=DEVICE)).cpu().numpy()
        corr = float(np.corrcoef(pred[:, 0], pred[:, 1])[0, 1])
        out[name] = {"corr": corr, "dim0_range": (float(pred[:,0].min()), float(pred[:,0].max())),
                      "dim1_range": (float(pred[:,1].min()), float(pred[:,1].max())), "pred": pred}
        print(f"{name} (UNTRAINED, T={T}): corr={corr:.3f}  dim0_range={out[name]['dim0_range']}  dim1_range={out[name]['dim1_range']}")
    return out

pareto_out = pareto_correlation_check(base_weights, afferent, efferent)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (name, d) in zip(axes, pareto_out.items()):
    p = d["pred"]
    ax.scatter(p[:, 0], p[:, 1], s=2, alpha=0.3)
    ax.set_title(f"{name}\ncorr={d['corr']:.3f}")
    ax.set_xlabel("boyut 0 (\"enerji\")"); ax.set_ylabel("boyut 1 (\"odul\")")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/pareto_null_controlled.png", dpi=140); plt.show()

with open(f"{OUT_DIR}/pareto_null_controlled_summary.json", "w") as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != "pred"} for k, v in pareto_out.items()}, f, indent=2)

diff = abs(pareto_out["real_connectome"]["corr"] - pareto_out["degree_preserving_null"]["corr"])
print(f"\nSONUC: real corr={pareto_out['real_connectome']['corr']:.3f} vs null corr={pareto_out['degree_preserving_null']['corr']:.3f} (fark={diff:.3f})")
print("Eger fark kucukse (<0.1-0.2), korelasyon 'biyolojik' degil, 2D dogrusal okuma katmaninin genel bir ozelligidir.")

## 5. Özet rapor + zip

In [ ]:
summary = {
    "saglik_kontrolu": {k: v for k, v in sanity.items() if k not in ('real_values','null_values')},
    "subgraf_tohum_taramasi": {
        "n_tohum": len(df_sweep), "pass_orani": f"{df_sweep['gate_pass'].sum()}/{len(df_sweep)}",
        "delta_medyan": float(df_sweep["cliffs_delta"].median()),
        "delta_min": float(df_sweep["cliffs_delta"].min()), "delta_max": float(df_sweep["cliffs_delta"].max()),
    },
    "T_duyarliligi": df_t.to_dict(orient="records"),
    "pareto_null_kontrollu": {k: {kk: vv for kk, vv in v.items() if kk != "pred"} for k, v in pareto_out.items()},
}
with open(f"{OUT_DIR}/OZET_RAPOR.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
import shutil
zip_path = shutil.make_archive("flyopt_verification_results", "zip", OUT_DIR)
print("zip hazir:", zip_path)
from google.colab import files
files.download(zip_path)